<a href="https://colab.research.google.com/github/kush951/NLP/blob/main/text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [10]:
df = pd.read_csv('/content/IMDB Dataset.csv', engine='python', on_bad_lines='skip')
display(df.head())

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [11]:
df = df.iloc[:10000]

In [14]:
df['sentiment'].value_counts()

,count
sentiment,
positive,5028
negative,4972


In [15]:
# Basic Preprocessing
# Remove tags
# lowercase
# remove stopwords

In [16]:
import re
def remove_tags(raw_text):
    cleaned_text = re.sub(re.compile('<.*?>'), '', raw_text)
    return cleaned_text

In [18]:
df['review'] = df['review'].apply(remove_tags)

In [19]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. The filming tec...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
9995,"Fun, entertaining movie about WWII German spy ...",positive
9996,Give me a break. How can anyone say that this ...,negative
9997,This movie is a bad movie. But after watching ...,negative
9998,This is a movie that was probably made to ente...,negative


In [20]:
df['review'] = df['review'].apply(lambda x:x.lower())

In [22]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

sw_list = stopwords.words('english')

df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [23]:
df

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production. filming technique...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically there's family little boy (jake) thi...,negative
4,"petter mattei's ""love time money"" visually stu...",positive
...,...,...
9995,"fun, entertaining movie wwii german spy (julie...",positive
9996,"give break. anyone say ""good hockey movie""? kn...",negative
9997,movie bad movie. watching endless series bad h...,negative
9998,"movie probably made entertain middle school, e...",negative


In [24]:
X = df.iloc[:,0:1]
y = df['sentiment']

In [25]:

from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(y)

In [26]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [27]:
X_train.shape

(8000, 1)

In [28]:
# Applying BoW
from sklearn.feature_extraction.text import CountVectorizer

# **Bag Of Words**

In [29]:
cv = CountVectorizer()
X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.transform(X_test['review']).toarray()

In [30]:
X_train_bow.shape

(8000, 48300)

In [31]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()

gnb.fit(X_train_bow,y_train)

GaussianNB()

In [32]:
y_pred = gnb.predict(X_test_bow)

from sklearn.metrics import accuracy_score,confusion_matrix
accuracy_score(y_test,y_pred)

0.6325

In [33]:
confusion_matrix(y_test,y_pred)

array([[721, 257],
       [478, 544]])

In [34]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()

rf.fit(X_train_bow,y_train)
y_pred = rf.predict(X_test_bow)
accuracy_score(y_test,y_pred)

0.8635

In [35]:
cv = CountVectorizer(max_features=3000) #hyperparameter Tunning

X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.transform(X_test['review']).toarray()

rf = RandomForestClassifier()

rf.fit(X_train_bow,y_train)
y_pred = rf.predict(X_test_bow)
accuracy_score(y_test,y_pred)

0.846

In [36]:
cv = CountVectorizer(ngram_range=(1,2),max_features=5000) #hyperparameter Tunning and N-Grams

X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.transform(X_test['review']).toarray()

rf = RandomForestClassifier()

rf.fit(X_train_bow,y_train)
y_pred = rf.predict(X_test_bow)
accuracy_score(y_test,y_pred)

0.8605

# **Tf-Idf**

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [38]:
tfidf = TfidfVectorizer()

In [39]:
X_train_tfidf = tfidf.fit_transform(X_train['review']).toarray()
X_test_tfidf = tfidf.transform(X_test['review'])

In [40]:
rf = RandomForestClassifier()

rf.fit(X_train_tfidf,y_train)
y_pred = rf.predict(X_test_tfidf)

accuracy_score(y_test,y_pred)

0.849

In [42]:
!pip install gensim
import gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 31.1 MB/s eta 0:00:00


In [47]:
import nltk
nltk.download('punkt_tab')
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [48]:
story = []
for doc in df['review']:
    raw_sent = sent_tokenize(doc)
    for sent in raw_sent:
        story.append(simple_preprocess(sent))

In [49]:
model = gensim.models.Word2Vec(
    window=10,
    min_count=2
)

In [50]:
model.build_vocab(story)
model.train(story, total_examples=model.corpus_count, epochs=model.epochs)

(5859411, 6196785)

In [51]:
len(model.wv.index_to_key)

31883

In [54]:
import numpy as np
def document_vector(doc):
    # remove out-of-vocabulary words
    doc = [word for word in doc.split() if word in model.wv.index_to_key]
    return np.mean(model.wv[doc], axis=0)

In [55]:
document_vector(df['review'].values[0])

array([ 0.08536918,  0.26054123,  0.16626185, -0.03971818, -0.01284105,
       -0.58362913,  0.2351282 ,  0.7567752 , -0.36433053,  0.01972824,
       -0.1985216 , -0.51845646,  0.09955481,  0.17877541,  0.16615954,
       -0.27981737,  0.16005571, -0.59285486, -0.01037686, -0.7029478 ,
        0.2211863 ,  0.16022557,  0.14992937, -0.36786562, -0.10235119,
       -0.1623929 , -0.16754249, -0.3644367 , -0.25499707,  0.02623362,
        0.3607609 , -0.01104469,  0.07502754, -0.32649133, -0.4025586 ,
        0.35091898,  0.02861097, -0.3930131 , -0.2895452 , -0.605906  ,
        0.04361869, -0.18447019, -0.09601173, -0.02591002,  0.23780492,
       -0.09108882, -0.23490027, -0.05639359,  0.16874762,  0.20147078,
        0.09860234, -0.22373717, -0.3369623 , -0.11294224, -0.30301172,
        0.16527823,  0.18345916,  0.12656482, -0.2806763 ,  0.08517208,
        0.00191642,  0.10692538, -0.08209972, -0.10677019, -0.27535224,
        0.36968067,  0.06113778,  0.16058306, -0.5766115 ,  0.21

In [56]:
from tqdm import tqdm

In [57]:
X = []
for doc in tqdm(df['review'].values):
    X.append(document_vector(doc))

100%|██████████| 10000/10000 [09:19<00:00, 17.86it/s]


In [58]:
X = np.array(X)

In [59]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

y = encoder.fit_transform(df['sentiment'])

In [60]:

y

array([1, 1, 1, ..., 0, 0, 1])

In [61]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [62]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [63]:
rf = RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test,y_pred)

0.77